In [2]:
!pip install openmeteo_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.8/707.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.5 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [3]:
import openmeteo_requests
from datetime import datetime

class IncreaseSpeed:

    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        # Stop iteration if max speed is already reached
        if self.current_speed >= self.max_speed:
            raise StopIteration

        # Calculate new speed, snapping to max_speed if step exceeds it
        new_speed = self.current_speed + self.step
        if new_speed > self.max_speed:
            self.current_speed = self.max_speed
        else:
            self.current_speed = new_speed

        return self.current_speed

class DecreaseSpeed:

    def __init__(self, current_speed: int, min_speed: int, step=10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        # Stop iteration if min speed is already reached
        if self.current_speed <= self.min_speed:
            raise StopIteration

        # Calculate new speed, snapping to min_speed if step exceeds it
        new_speed = self.current_speed - self.step
        if new_speed < self.min_speed:
            self.current_speed = self.min_speed
        else:
            self.current_speed = new_speed

        return self.current_speed

class Car:

    # Class variable to track total cars on the road
    total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed
        self.on_road = True
        Car.total_cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):
        if not self.on_road:
            print("The car is parked and cannot accelerate.")
            return f"Current speed: {self.current_speed} km/h"

        # Determine the target speed limit
        if upper_border is None:

            target = min(self.current_speed + step, self.max_speed)
        else:

            target = min(upper_border, self.max_speed)
            if target <= self.current_speed:
                print("Target speed must be higher than current speed.")
                return f"Current speed: {self.current_speed} km/h"

        increaser = IncreaseSpeed(self.current_speed, target, step)

        # Iterate over the increaser until target is met
        for speed in increaser:
            self.current_speed = speed
            print(f"Accelerating... Current speed: {self.current_speed} km/h")

        return f"Final current speed: {self.current_speed} km/h"

    def brake(self, lower_border=None, step=10):
        if not self.on_road:
            print("The car is parked.")
            return f"Current speed: {self.current_speed} km/h"

        # Determine the target speed limit
        if lower_border is None:
            # If no border passed, just decrease once (down to 0)
            target = max(self.current_speed - step, 0)
        else:
            # Ensure lower_border does not go below 0
            target = max(lower_border, 0)
            if target >= self.current_speed:
                print("Target speed must be lower than current speed.")
                return f"Current speed: {self.current_speed} km/h"

        decreaser = DecreaseSpeed(self.current_speed, target, step)

        # Iterate over the decreaser until target is met
        for speed in decreaser:
            self.current_speed = speed
            print(f"Braking... Current speed: {self.current_speed} km/h")

        return f"Final current speed: {self.current_speed} km/h"

    # 1. Regular method: modifies instance state and class variable
    def parking(self):
        if self.on_road:
            self.on_road = False
            self.current_speed = 0
            Car.total_cars_on_road -= 1
            print("The car is now parked off the road.")
        else:
            print("The car is already parked.")

    # 2. Class method: accesses class variable only
    @classmethod
    def total_cars(cls):
        print(f"Total amount of cars on the road: {cls.total_cars_on_road}")
        return cls.total_cars_on_road

    # 3. Static method: independent utility function, doesn't need class or instance state
    @staticmethod
    def show_weather():
        openmeteo = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }

        try:
            response = openmeteo.weather_api(url, params=params)[0]
            current = response.Current()

            print(f"Current time: {datetime.fromtimestamp(current.Time() + response.UtcOffsetSeconds())} {response.TimezoneAbbreviation().decode()}")
            print(f"Current temperature: {round(current.Variables(0).Value(), 0)} C")
            print(f"Current apparent_temperature: {round(current.Variables(1).Value(), 0)} C")
            print(f"Current rain: {current.Variables(2).Value()} mm")
            print(f"Current wind_speed: {round(current.Variables(3).Value(), 1)} m/s")
        except Exception as e:
            print(f"Could not retrieve weather data. Error: {e}")

In [7]:
# 1. Instantiate objects
my_car = Car(max_speed=60)
Car(max_speed=100) # Instantiate a second car to verify the total number of cars on the road

# 2. Test the three different method types
print("--- Method Types Test ---")
Car.show_weather()  # Static method @staticmethod
Car.total_cars()    # Class method @classmethod

# 3. Test iterators (acceleration & braking) and state control
print("\n--- Iterators and State Test ---")
my_car.accelerate(upper_border=30, step=15) # Test IncreaseSpeed iterator
my_car.brake(lower_border=10, step=10)      # Test DecreaseSpeed iterator

my_car.parking()                            # Test regular method (changes instance state)
my_car.accelerate(step=10)                  # Verify state interception after parking

--- Method Types Test ---
Current time: 2026-03-18 21:15:00 GMT+3
Current temperature: 3.0 C
Current apparent_temperature: -0.0 C
Current rain: 0.0 mm
Current wind_speed: 3.3 m/s
Total amount of cars on the road: 4

--- Iterators and State Test ---
Accelerating... Current speed: 15 km/h
Accelerating... Current speed: 30 km/h
Braking... Current speed: 20 km/h
Braking... Current speed: 10 km/h
The car is now parked off the road.
The car is parked and cannot accelerate.


'Current speed: 0 km/h'